# MiniEngine on Google Colab

End-to-end runner for the **CS349D MiniEngine** repo. This notebook:

1. Verifies a GPU is attached
2. Clones the (public) GitHub repo
3. Installs base dependencies
4. Installs `flash-attn` from a prebuilt wheel (needed for `--mode paged`)
5. Loads your HuggingFace token from Colab secrets (for faster downloads)
6. Launches the OpenAI-compatible server in the background
7. Sends a sample chat request to verify it works
8. (Optional) Runs a quick serving / accuracy benchmark
9. Cleanly shuts the server down

## Before you start

**Enable GPU**: `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is free and is enough for `Qwen/Qwen3-4B-Instruct-2507` in bf16).

**Add your HuggingFace token to Colab secrets** (recommended — avoids the "unauthenticated requests" warning and rate limits):
1. Get a token at https://huggingface.co/settings/tokens (read access is enough).
2. In Colab, click the **🔑 key icon** in the left sidebar → **Add new secret**.
3. **Name:** `HF_TOKEN` &nbsp;&nbsp;**Value:** your token.
4. Toggle **Notebook access** on for this notebook.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the repository

The repo is public, so no authentication is needed.

In [ ]:
import os
import shutil
import subprocess

GITHUB_USER = "bwathomas"  # @param {type:"string"}
REPO_NAME   = "MAST-miniengine-BWT"  # @param {type:"string"}
BRANCH      = "MAST-miniengine-BWT"  # @param {type:"string"}

WORKDIR = f"/content/{REPO_NAME}"

if os.path.isdir(WORKDIR):
    print(f"Removing existing checkout at {WORKDIR} \u2026")
    shutil.rmtree(WORKDIR)

clone_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, WORKDIR],
    check=True,
)

%cd $WORKDIR
!git log --oneline -5

## 3. Install base dependencies

Installs `miniengine` itself plus the benchmark extras. PyTorch is preinstalled on Colab so this is mostly small packages.

`flash-attn` is **not** installed here — it's a separate step (next cell) because it needs a prebuilt wheel.

In [ ]:
!pip install -q -e .
!pip install -q -e ".[bench]" datasets requests

## 4. Install `flash-attn` from a prebuilt wheel

`flash-attn` is required for **`--mode paged`** (Milestone 2 Part B). Building it from source on a fresh VM takes 30–60 min — but the maintainer ships **prebuilt wheels** for common torch/CUDA/Python combinations that install in ~30 seconds.

This cell:
1. Detects your torch / CUDA / Python versions.
2. Builds the matching wheel URL from the [flash-attention releases](https://github.com/Dao-AILab/flash-attention/releases).
3. Probes a few recent versions to find one with a wheel for your config.
4. `pip install`s it directly from the URL.

**Skip this cell if you only plan to run `--mode batched` (Milestone 1).** The server will still launch without flash-attn; you just can't pass `--mode paged`.

In [ ]:
import subprocess
import sys
import urllib.error
import urllib.request

import torch

# Dao-AILab/flash-attention release tags, newest first. Each release
# covers a set of (torch_short, cuda_major, py_short, abi) wheels; the
# probe stops at the first hit.
_FA_VERSIONS = [
    "2.8.1",
    "2.8.0.post2", "2.8.0",
    "2.7.4.post1", "2.7.3", "2.6.3",
    "2.5.9.post1",
]

torch_full = torch.__version__.split("+")[0]
torch_short = ".".join(torch_full.split(".")[:2])
cuda_major = (torch.version.cuda or "").split(".")[0]
py_short = f"cp{sys.version_info.major}{sys.version_info.minor}"
# Torch reports its own ABI; honor that first, try the opposite as a fallback.
abi_detected = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
abi_order = [abi_detected, "FALSE" if abi_detected == "TRUE" else "TRUE"]

print(
    f"Detected:  torch={torch_full}  cuda={torch.version.cuda}  "
    f"py={py_short}  abi={abi_detected}"
)

if not cuda_major:
    raise RuntimeError(
        "torch.version.cuda is empty \u2014 your torch build doesn't have CUDA. "
        "Change Colab runtime to GPU and restart."
    )

def _wheel_url(fa_v: str, abi: str) -> str:
    return (
        "https://github.com/Dao-AILab/flash-attention/releases/download/"
        f"v{fa_v}/flash_attn-{fa_v}+cu{cuda_major}torch{torch_short}cxx11abi{abi}"
        f"-{py_short}-{py_short}-linux_x86_64.whl"
    )

def _url_exists(url: str, timeout: float = 10.0) -> bool:
    req = urllib.request.Request(url, method="HEAD")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return 200 <= r.status < 400
    except urllib.error.HTTPError as e:
        return 200 <= e.code < 400  # treat 200/302 as success, 404 as miss
    except Exception:
        return False

wheel_url = None
for fa_v in _FA_VERSIONS:
    for abi in abi_order:
        candidate = _wheel_url(fa_v, abi)
        print(f"  probing  flash-attn=={fa_v}  abi={abi}  \u2026", end=" ")
        if _url_exists(candidate):
            wheel_url = candidate
            print("OK")
            break
        print("miss")
    if wheel_url:
        break

if wheel_url:
    print(f"\nInstalling:\n  {wheel_url}\n")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", wheel_url]
    )
    import importlib
    importlib.invalidate_caches()
    import flash_attn  # type: ignore[import-not-found]
    print(f"flash-attn {flash_attn.__version__} installed.")
else:
    print(
        "\nNo prebuilt wheel matched this environment.\n"
        f"  torch={torch_full}  cuda={torch.version.cuda}  py={py_short}  "
        f"abi={abi_detected}\n\nOptions:\n"
        "  1. Browse https://github.com/Dao-AILab/flash-attention/releases\n"
        "     and pip install a matching URL manually.\n"
        "  2. Pin torch to a version with wheel coverage, e.g.:\n"
        "       pip install --upgrade 'torch==2.7.*'\n"
        "     then restart the runtime and re-run this cell.\n"
        "  3. Build from source (30\u201360 min):\n"
        "       pip install flash-attn --no-build-isolation\n"
        "  4. Skip flash-attn and use --mode batched only (M1 still works)."
    )

## 5. Load HuggingFace token from Colab secrets

Pulls the `HF_TOKEN` secret you set in the sidebar and exports it to the environment. This eliminates the "unauthenticated requests" warning and unlocks higher download rate limits.

If the secret isn't set, the cell warns and continues — anonymous downloads still work, just slower.

In [ ]:
def _load_hf_token() -> str | None:
    try:
        from google.colab import userdata
    except ImportError:
        return os.environ.get("HF_TOKEN")
    try:
        return userdata.get("HF_TOKEN")
    except Exception as e:
        print(f"  No HF_TOKEN secret available ({type(e).__name__}: {e}).")
        return None

hf_token = _load_hf_token()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    masked = hf_token[:4] + "\u2026" + hf_token[-4:] if len(hf_token) > 8 else "set"
    print(f"  HF token loaded ({masked}).")
else:
    print("  Proceeding without an HF token \u2014 downloads may be rate-limited.")

### (Optional) Persist the HuggingFace model cache to Google Drive

Qwen3-4B weights are ~8 GB. Colab wipes `/root/.cache` between sessions, so subsequent runs would re-download. Mount Drive and point `HF_HOME` there to cache across sessions. **Skip this cell if you don't care about repeat-run speed.**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME =", os.environ["HF_HOME"])

## 6. Launch the server in the background

Starts `python -m miniengine` as a subprocess, redirecting all logs to `server.log`. Then polls `/health` until the server is responsive. **The model will download on first run (~8 GB), so the first wait can take 3–5 minutes.**

Change `MODEL` to use a different one. Suggestions:
- `Qwen/Qwen3-0.6B-Instruct-2507` — fastest, for smoke tests
- `Qwen/Qwen3-4B-Instruct-2507` — README default, good for T4
- `Qwen/Qwen3-8B` — milestone-2 target, needs L4 or better

Set `MODE = "paged"` to exercise Milestone 2 (requires flash-attn from step 4).

In [ ]:
import signal
import subprocess
import time

import requests

MODEL   = "Qwen/Qwen3-4B-Instruct-2507"  # @param {type:"string"}
DTYPE   = "bfloat16"                      # @param ["bfloat16", "float16", "float32"]
MODE    = "batched"                       # @param ["baseline", "batched", "paged"]
PORT    = 8000
MAX_RUN = 16

log_path = os.path.join(WORKDIR, "server.log")
log_f = open(log_path, "w")

server_env = os.environ.copy()

server_proc = subprocess.Popen(
    [
        "python", "-u", "-m", "miniengine",
        "--model", MODEL,
        "--dtype", DTYPE,
        "--mode", MODE,
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "--max-running", str(MAX_RUN),
    ],
    stdout=log_f,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    cwd=WORKDIR,
    env=server_env,
)
print(f"Server PID = {server_proc.pid}.  Logs \u2192 {log_path}")

BASE_URL = f"http://127.0.0.1:{PORT}"
DEADLINE = time.time() + 600

while True:
    if server_proc.poll() is not None:
        print("\nServer crashed. Last 60 log lines:")
        print(subprocess.check_output(["tail", "-n", "60", log_path]).decode())
        raise RuntimeError("miniengine exited before becoming healthy")
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=2)
        if r.status_code == 200:
            print("\nServer is healthy.")
            break
    except requests.exceptions.RequestException:
        pass
    if time.time() > DEADLINE:
        raise TimeoutError("Server did not become healthy within 10 minutes")
    print(".", end="", flush=True)
    time.sleep(5)

### Peek at the server log (handy while waiting / after a crash)

In [ ]:
!tail -n 40 server.log

## 7. Send a sample chat request

In [ ]:
import json

payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "In one sentence, explain why batched decoding speeds up an LLM serving engine."}
    ],
    "max_tokens": 128,
    "temperature": 0.0,
    "stream": True,
}

print("\nStreaming response:\n")
with requests.post(f"{BASE_URL}/v1/chat/completions", json=payload, stream=True, timeout=120) as resp:
    resp.raise_for_status()
    for raw in resp.iter_lines():
        if not raw:
            continue
        line = raw.decode("utf-8")
        if not line.startswith("data: "):
            continue
        data = line[len("data: "):]
        if data == "[DONE]":
            print("\n\n[stream finished]")
            break
        chunk = json.loads(data)
        delta = chunk["choices"][0].get("delta", {})
        piece = delta.get("content", "")
        print(piece, end="", flush=True)

## 8. (Optional) Quick serving benchmark

Tiny configuration so it finishes in a few minutes. For a real measurement, increase `--num-requests` and add more concurrency levels.

In [ ]:
!python -m benchmark.bench_serving \
    --input-len 256 --output-len 128 \
    --concurrencies 1,4,8 \
    --num-requests 8 \
    --randomness 0.5

## 9. (Optional) Quick accuracy benchmark

In [ ]:
!python -m benchmark.bench_accuracy --dataset mmlu --num-samples 50 --concurrency 4

## 10. Stop the server

Always run this when you're done so the GPU memory is released and the runtime is reusable.

In [ ]:
if server_proc.poll() is None:
    try:
        os.killpg(os.getpgid(server_proc.pid), signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        server_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        os.killpg(os.getpgid(server_proc.pid), signal.SIGKILL)
        server_proc.wait()
    print("Server stopped.")
else:
    print("Server already exited with code", server_proc.returncode)

log_f.close()

## 11. Milestone 2 benchmark suite

Runs the **full set of measurements required by the M2 PDF report**:

1. **Throughput sweep** — same workload across 4 configs:
   M1 `batched` → M2 `paged` → `paged + torch.compile` → `paged + torch.compile + cuda-graph`
2. **Page-size comparison** — `--page-size 256` vs `512` (flash-attn 2.8+ requires page_size to be a multiple of 256, so the comparison is between the two smallest valid values)
3. **Accuracy parity** — MMLU on `paged` and `paged + torch.compile`

All outputs go to `bench_results/` in the cloned repo. The final cell prints a compact summary so you can sanity-check before screenshotting for the PDF.

**Hardware notes**
- **L4 / A100** (Colab Pro): keep `MODEL_M2 = "Qwen/Qwen3-8B"` (default — matches the rubric).
- **T4** (free Colab): switch to `Qwen/Qwen3-4B-Instruct-2507` and `MAX_RUN = 16`. Qwen3-8B won't fit in T4's 16 GB VRAM. **Note the model substitution in your report.**

**Runtime**
- L4 + Qwen3-8B: 30–60 min end-to-end.
- T4 + Qwen3-4B: 15–30 min.

**Prerequisites**
- Sections 1–4 must have run successfully (GPU, clone, install, flash-attn).
- This section is **self-contained** — it stops any server you started in section 6 and manages its own.

### 11.1 Setup — stop the demo server, set config

`MODEL_M2`, `MAX_RUN`, and `PAGE_SIZE` are the main knobs. The **workload params** below (`BENCH_INPUT_LEN`, etc.) are kept IDENTICAL across all rows for a fair comparison — don't change them between rows.

In [ ]:
import signal
import subprocess
import time

import requests


try:
    if server_proc.poll() is None:
        os.killpg(os.getpgid(server_proc.pid), signal.SIGTERM)
        try:
            server_proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(os.getpgid(server_proc.pid), signal.SIGKILL)
            server_proc.wait()
        log_f.close()
        print("Stopped server from section 6.")
        time.sleep(3)
except NameError:
    pass

MODEL_M2  = "Qwen/Qwen3-8B"                  # @param {type:"string"}
DTYPE_M2  = "bfloat16"                       # @param ["bfloat16", "float16"]
MAX_RUN   = 32                               # @param {type:"integer"}
MEM_FRAC  = 0.85                             # @param {type:"number"}
# flash-attn 2.8+ requires page_size % 256 == 0. Valid values: 256, 512, 1024.
PAGE_SIZE = 256                              # @param {type:"integer"}

# Workload — kept identical across rows for fair comparison.
# - INPUT/OUTPUT 512/512: decode-heavy, which is where paged shines vs M1.
# - NUM_REQS=64 fills conc=64 in one full wave (queue depth 32 vs
#   max_running=32 — exercises the scheduler's admission control).
# - Concurrencies skip conc=1 deliberately: with 64 sequential requests
#   it adds ~60 min to the sweep for a single-stream latency datapoint
#   that doesn't validate any milestone target. The three targets
#   (2x paged, +10% compile, +20% cudagraph) are all at high concurrency.
#   If you need single-stream numbers for the report, run a separate
#   small conc=1 bench after the sweep.
#     8   : 8 waves of 8 = stable p50/p99 stats
#     32  : exactly fills max_running (2 waves of 32)
#     64  : 2x oversubscribed, exercises the queue + admission control
BENCH_INPUT_LEN  = 512
BENCH_OUTPUT_LEN = 512
BENCH_CONC       = "8,32,64"
BENCH_NUM_REQS   = 64
BENCH_RANDOMNESS = 0.5

RESULTS_DIR = os.path.join(WORKDIR, "bench_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results directory: {RESULTS_DIR}")
print(f"Model: {MODEL_M2}  dtype: {DTYPE_M2}  max-running: {MAX_RUN}")
print(f"Workload: in={BENCH_INPUT_LEN} out={BENCH_OUTPUT_LEN} "
      f"conc={BENCH_CONC} num={BENCH_NUM_REQS}")

### 11.2 Server lifecycle helpers

Each row of the sweep needs a fresh server (so `torch.compile` / CUDA graphs / KV pool state don't leak between rows). The helpers below start the server, wait for `/health`, run the benchmark, then cleanly shut down.

The `warmup_server()` call between health and benchmark fires one tiny request — important for `--torch-compile` runs because Dynamo traces on the first forward pass at each new shape, and we don't want that compile cost polluting the first measurement.

In [ ]:
def _start_server(extra_args, log_name):
    log_path = os.path.join(RESULTS_DIR, log_name)
    log_f = open(log_path, "w")
    cmd = [
        "python", "-u", "-m", "miniengine",
        "--model", MODEL_M2,
        "--dtype", DTYPE_M2,
        "--host", "127.0.0.1", "--port", "8000",
        "--max-running", str(MAX_RUN),
    ] + extra_args
    proc = subprocess.Popen(
        cmd, stdout=log_f, stderr=subprocess.STDOUT,
        start_new_session=True, cwd=WORKDIR, env=os.environ.copy(),
    )
    return proc, log_f, log_path


def _wait_for_health(proc, log_path, timeout=900):
    deadline = time.time() + timeout
    print("    waiting for /health", end="", flush=True)
    while True:
        if proc.poll() is not None:
            tail = subprocess.check_output(["tail", "-n", "60", log_path]).decode()
            print(f"\n[server log tail]\n{tail}")
            raise RuntimeError("server exited before becoming healthy")
        try:
            r = requests.get("http://127.0.0.1:8000/health", timeout=2)
            if r.status_code == 200:
                print(" healthy.")
                return
        except requests.exceptions.RequestException:
            pass
        if time.time() > deadline:
            raise TimeoutError("/health did not respond within 15 min")
        print(".", end="", flush=True)
        time.sleep(5)


def _stop_server(proc, log_f):
    if proc.poll() is None:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            proc.wait()
    log_f.close()
    time.sleep(3)


def _run_subproc_tee(cmd, capture_to):
    print(f"    output -> {capture_to}")
    with open(capture_to, "w") as fout:
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            cwd=WORKDIR, text=True,
        )
        for line in proc.stdout:
            fout.write(line)
            print("    ", line.rstrip())
        proc.wait()
    return proc.returncode


def warmup_server():
    try:
        requests.post(
            "http://127.0.0.1:8000/v1/chat/completions",
            json={
                "model": "x",
                "messages": [{"role": "user", "content": "hi"}],
                "max_tokens": 4, "temperature": 0, "stream": False,
            }, timeout=300,
        )
    except Exception:
        pass


def run_serving(label, server_args):
    print(f"\n{'=' * 80}\n  ROW: {label}\n{'=' * 80}")
    print(f"  server args: {server_args}")
    proc, log_f, log_path = _start_server(server_args, f"server_{label}.log")
    try:
        _wait_for_health(proc, log_path)
        print("    warming up...", end="", flush=True)
        warmup_server()
        print(" done.")
        out_path = os.path.join(RESULTS_DIR, f"bench_{label}.txt")
        _run_subproc_tee(
            [
                "python", "-m", "benchmark.bench_serving",
                "--input-len", str(BENCH_INPUT_LEN),
                "--output-len", str(BENCH_OUTPUT_LEN),
                "--randomness", str(BENCH_RANDOMNESS),
                "--concurrencies", BENCH_CONC,
                "--num-requests", str(BENCH_NUM_REQS),
            ],
            out_path,
        )
        return out_path
    finally:
        _stop_server(proc, log_f)


def run_accuracy(label, server_args, dataset="mmlu", num_samples=200):
    print(f"\n{'=' * 80}\n  ACCURACY: {label} ({dataset}, n={num_samples})\n{'=' * 80}")
    proc, log_f, log_path = _start_server(server_args, f"server_acc_{label}.log")
    try:
        _wait_for_health(proc, log_path)
        out_path = os.path.join(RESULTS_DIR, f"accuracy_{dataset}_{label}.txt")
        _run_subproc_tee(
            [
                "python", "-m", "benchmark.bench_accuracy",
                "--dataset", dataset,
                "--num-samples", str(num_samples),
                "--concurrency", "4",
            ],
            out_path,
        )
        return out_path
    finally:
        _stop_server(proc, log_f)

print("helpers ready.")

### 11.3 Throughput sweep (4 rows)

Headline metric is **GenTok/s** from `bench_serving`. Targets per the milestone:

| Row | Target |
|---|---|
| Row 2 (paged) vs Row 1 (M1 batched) | ≥ 2× throughput |
| Row 3 (paged + compile) vs Row 2 | ≥ 10% |
| Row 4 (+ cuda-graph) vs Row 3 | ≥ 20% |

This cell runs all four rows back-to-back. **Expected: 20–40 min on L4.**

In [ ]:
SWEEP = [
    ("01_m1_batched", [
        "--mode", "batched",
    ]),
    ("02_m2_paged", [
        "--mode", "paged",
        "--mem-fraction-static", str(MEM_FRAC),
        "--page-size", str(PAGE_SIZE),
    ]),
    ("03_paged_compile", [
        "--mode", "paged",
        "--mem-fraction-static", str(MEM_FRAC),
        "--page-size", str(PAGE_SIZE),
        "--torch-compile",
    ]),
    ("04_paged_compile_cudagraph", [
        "--mode", "paged",
        "--mem-fraction-static", str(MEM_FRAC),
        "--page-size", str(PAGE_SIZE),
        "--torch-compile",
        "--cuda-graph",
        "--cuda-graph-batch-sizes", "1,2,4,8,16,32",
    ]),
]

sweep_results = {}
for label, args in SWEEP:
    sweep_results[label] = run_serving(label, args)

print("\nThroughput sweep complete. Output files:")
for label, path in sweep_results.items():
    print(f"  {label:<32} {path}")

### 11.4 Page-size comparison

Two paged runs with `--page-size 256` and `--page-size 512`. Same workload as the throughput sweep, only `--page-size` changes.

**Note for the report:** the milestone suggests "e.g., 16 vs 128", but flash-attn 2.8+ requires the paged KV block size to be a multiple of 256, so the comparison is between the two smallest valid values. Trade-off direction is the same: smaller pages → less wasted KV at sequence tails, bigger page tables; larger pages → opposite. Effect on raw throughput is usually small.

In [ ]:
PAGE_RUNS = [
    ("page256", ["--mode", "paged", "--mem-fraction-static", str(MEM_FRAC), "--page-size", "256"]),
    ("page512", ["--mode", "paged", "--mem-fraction-static", str(MEM_FRAC), "--page-size", "512"]),
]

page_results = {}
for label, args in PAGE_RUNS:
    page_results[label] = run_serving(label, args)

print("\nPage-size sweep complete. Output files:")
for label, path in page_results.items():
    print(f"  {label:<10} {path}")

### 11.5 Accuracy parity (MMLU)

`paged` vs `paged + torch.compile` on 200 MMLU samples. The two numbers must be within ~2 percentage points; bigger gaps mean `torch.compile` silently produced different numerics and need investigating before submission.

In [ ]:
acc_results = {}
acc_results["paged"] = run_accuracy(
    "paged",
    ["--mode", "paged", "--mem-fraction-static", str(MEM_FRAC), "--page-size", str(PAGE_SIZE)],
    dataset="mmlu", num_samples=200,
)
acc_results["paged_compile"] = run_accuracy(
    "paged_compile",
    ["--mode", "paged", "--mem-fraction-static", str(MEM_FRAC),
     "--page-size", str(PAGE_SIZE), "--torch-compile"],
    dataset="mmlu", num_samples=200,
)

print("\nAccuracy runs complete. Output files:")
for label, path in acc_results.items():
    print(f"  {label:<14} {path}")

### 11.6 Results summary

Prints the final summary tables from each `bench_*.txt` and the headline numbers from each `accuracy_*.txt`. Use these for the PDF report; the raw output files (with per-request progress) are also in `bench_results/` if you want to download them all.

In [ ]:
import glob

print("=" * 88)
print("Milestone 2 results summary")
print(f"  Results directory: {RESULTS_DIR}")
print("=" * 88)

for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "bench_*.txt"))):
    print(f"\n--- {os.path.basename(path)} ---")
    with open(path) as f:
        text = f.read()
    lines = text.splitlines()
    start = None
    for i, line in enumerate(lines):
        if line.startswith("=" * 50) or line.startswith("=" * 60):
            start = i
            break
    print("\n".join(lines[start:] if start is not None else lines[-30:]))

for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "accuracy_*.txt"))):
    print(f"\n--- {os.path.basename(path)} ---")
    with open(path) as f:
        for line in f:
            stripped = line.strip()
            if any(k in stripped for k in ("Accuracy", "Samples", "Correct", "Avg latency")):
                print(" ", stripped)

print("\nDone. Screenshot the above sections for the PDF report.")